# Appending generated NetCDF slices

This example uses the same daily synthetic data as `zarr_v2.ipynb`, but writes the input slices as NetCDF files. The two `zappend` calls deliberately match the v2 example so the input format is the only difference.

In [8]:
import os
from pathlib import Path
import shutil

import xarray as xr

from zappend.api import zappend
from create_zarr_inputs import make_netcdf_slices

## Generate and collect the NetCDF inputs

`make_netcdf_slices` uses the shared `make_slice` data factory and writes compressed NetCDF files to `inputs/netcdf`.

In [2]:
%%time
make_netcdf_slices(31)

CPU times: user 4.15 s, sys: 725 ms, total: 4.87 s
Wall time: 5.23 s


In [9]:
input_dir = Path('inputs/netcdf')
output_dir = Path('outputs')
output_dir.mkdir(exist_ok=True)
slices = sorted(str(path) for path in input_dir.glob('*.nc'))
assert len(slices) == 31
slices[:3]

['inputs/netcdf/slice-2024-01-01.nc',
 'inputs/netcdf/slice-2024-01-02.nc',
 'inputs/netcdf/slice-2024-01-03.nc']

## Default layout: favor spatial access

This call is identical to the first v2 Zarr example. With no target format specified, zappend writes a Zarr v2 target and obtains its initial metadata and encoding from the first NetCDF slice.

In [4]:
%%time
target = output_dir / 'netcdf_example_space_opt.zarr'
count = zappend(
    slices,
    target_dir=str(target),
    force_new=True,
    logging='INFO',
)
assert count == len(slices)

2026-10-29 17:44:29,055 WARNING Setting 'force_new' is enabled. This will permanently delete existing targets (no rollback).
2026-10-29 17:44:29,056 WARNING Permanently deleting outputs/netcdf_example_space_opt.zarr
2026-10-29 17:44:29,131 INFO Opening slice dataset from inputs/netcdf/slice-2024-01-01.nc
2026-10-29 17:44:29,166 INFO Creating target dataset outputs/netcdf_example_space_opt.zarr
2026-10-29 17:44:29,308 INFO Transaction completed.
2026-10-29 17:44:29,309 INFO Slice dataset inputs/netcdf/slice-2024-01-01.nc closed
2026-10-29 17:44:29,319 INFO Opening slice dataset from inputs/netcdf/slice-2024-01-02.nc
2026-10-29 17:44:29,329 INFO Updating target dataset outputs/netcdf_example_space_opt.zarr
2026-10-29 17:44:29,500 INFO Transaction completed.
2026-10-29 17:44:29,501 INFO Slice dataset inputs/netcdf/slice-2024-01-02.nc closed
2026-10-29 17:44:29,512 INFO Opening slice dataset from inputs/netcdf/slice-2024-01-03.nc
2026-10-29 17:44:29,521 INFO Updating target dataset outputs

CPU times: user 6.7 s, sys: 1.7 s, total: 8.4 s
Wall time: 6.17 s


In [5]:
%%time
ds = xr.open_zarr(target)
ds

CPU times: user 21.3 ms, sys: 6.13 ms, total: 27.4 ms
Wall time: 22.1 ms


<xarray.Dataset> Size: 1GB
Dimensions:       (time: 31, lat: 1800, lon: 3600)
Coordinates:
  * time          (time) datetime64[ns] 248B 2024-01-01 ... 2024-01-31
  * lat           (lat) float32 7kB -89.95 -89.85 -89.75 ... 89.75 89.85 89.95
  * lon           (lon) float32 14kB -179.9 -179.9 -179.8 ... 179.7 179.8 179.9
Data variables:
    quality_flag  (time, lat, lon) uint8 201MB dask.array<chunksize=(1, 450, 450), meta=np.ndarray>
    temperature   (time, lat, lon) float32 804MB dask.array<chunksize=(1, 450, 450), meta=np.ndarray>
Attributes:
    title:    Zappend example slice

## Time-series layout

This is the same second call as the v2 Zarr example. The target has 31 time values in each data chunk, so extracting a point time series reads one temporal chunk per 90 by 90 spatial tile. As in the other examples, choose this temporal size from the expected append horizon because existing Zarr chunks cannot expand.

In [6]:
%%time
full_time_chunk = len(slices)
target = output_dir / 'netcdf_example_time_opt.zarr'
count = zappend(
    slices,
    target_dir=str(target),
    force_new=True,
    variables={
        'temperature': {'encoding': {'chunks': [full_time_chunk, 90, 90]}},
        'quality_flag': {'encoding': {'chunks': [full_time_chunk, 90, 90]}},
    },
    logging='INFO',
)
assert count == full_time_chunk

2026-10-29 17:44:35,304 WARNING Setting 'force_new' is enabled. This will permanently delete existing targets (no rollback).
2026-10-29 17:44:35,305 WARNING Permanently deleting outputs/netcdf_example_time_opt.zarr
2026-10-29 17:44:35,358 INFO Opening slice dataset from inputs/netcdf/slice-2024-01-01.nc
2026-10-29 17:44:35,368 INFO Creating target dataset outputs/netcdf_example_time_opt.zarr
2026-10-29 17:44:38,003 INFO Transaction completed.
2026-10-29 17:44:38,004 INFO Slice dataset inputs/netcdf/slice-2024-01-01.nc closed
2026-10-29 17:44:38,014 INFO Opening slice dataset from inputs/netcdf/slice-2024-01-02.nc
2026-10-29 17:44:38,024 INFO Updating target dataset outputs/netcdf_example_time_opt.zarr
2026-10-29 17:44:41,837 INFO Transaction completed.
2026-10-29 17:44:41,838 INFO Slice dataset inputs/netcdf/slice-2024-01-02.nc closed
2026-10-29 17:44:41,848 INFO Opening slice dataset from inputs/netcdf/slice-2024-01-03.nc
2026-10-29 17:44:41,857 INFO Updating target dataset outputs/ne

CPU times: user 2min 32s, sys: 44.9 s, total: 3min 17s
Wall time: 2min 8s


In [7]:
%%time
ds = xr.open_zarr(target)
ds

CPU times: user 21.9 ms, sys: 661 μs, total: 22.6 ms
Wall time: 18.6 ms


<xarray.Dataset> Size: 1GB
Dimensions:       (time: 31, lat: 1800, lon: 3600)
Coordinates:
  * time          (time) datetime64[ns] 248B 2024-01-01 ... 2024-01-31
  * lat           (lat) float32 7kB -89.95 -89.85 -89.75 ... 89.75 89.85 89.95
  * lon           (lon) float32 14kB -179.9 -179.9 -179.8 ... 179.7 179.8 179.9
Data variables:
    quality_flag  (time, lat, lon) uint8 201MB dask.array<chunksize=(31, 90, 90), meta=np.ndarray>
    temperature   (time, lat, lon) float32 804MB dask.array<chunksize=(31, 90, 90), meta=np.ndarray>
Attributes:
    title:    Zappend example slice

In [10]:
if os.path.exists(input_dir):
    shutil.rmtree(input_dir)